# 03 · Results for the paper

- **A.** SmolLM2 baseline (CausalCoT run on the easy file), rescored with the tolerant yes/no parser.
- **B.** Commonsense-alignment results from notebook 02: accuracy by variant (H1), P(Yes) shift on matched pairs (H1b), accuracy by rung (H2), answer bias, topology, failure graphs.
- **C.** Export: LaTeX table rows and PDF figures into the paper folder.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from causal_evaluation.evaluation.failure_graph import draw_failure_graph, failure_graph
from causal_evaluation.evaluation.stats import extract_yes_no
from causal_evaluation.evaluation.variant_analysis import (
    load_scored,
    nonsense_gap,
    p_yes_shift,
    paired_accuracy_gap,
    rate_table,
)

INTERIM = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim"))
PAPER_DIR = Path(os.environ.get("CE_PAPER_DIR", ROOT.parent / "tesispp" / "cadis-extended-abstract"))
FIG_DIR, TAB_DIR = PAPER_DIR / "figures", PAPER_DIR / "tables"
for folder in (FIG_DIR, TAB_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)


def short(model):
    return model.split("/")[-1].split(":")[0]

## A · SmolLM2 baseline, rescored

In [ ]:
sample_full = pd.read_json(INTERIM / "sample-full.jsonl", lines=True, dtype={"question_id": str})
responses = pd.read_json(INTERIM / "responses-smollm2-full.jsonl", lines=True, dtype={"question_id": str})
responses["text"] = responses["response"].map(lambda r: r.get("text") or "")
responses["error"] = responses["response"].map(lambda r: r.get("error"))

base = responses.merge(
    sample_full[["question_id", "answer", "causal_level", "meta"]], on="question_id", validate="one_to_one"
)
base["answer"] = base["answer"].str.strip().str.lower()
base["story_kind"] = base["meta"].map(lambda m: "nonsense" if str(m["story_id"]).startswith("nonsense") else "named")
base["parsed"] = base["text"].map(extract_yes_no)
base["status"] = np.select([base["error"].notna(), base["parsed"].isna()], ["error", "unparsed"], "parsed")
base["correct_exact"] = base["text"].str.strip().str.casefold() == base["answer"]
base["correct_parsed"] = base["parsed"] == base["answer"]
base["is_yes"] = base["parsed"] == "yes"

base["status"].value_counts()

In [ ]:
baseline = pd.DataFrame(
    {
        "exact_match": base.groupby("causal_level")["correct_exact"].mean(),
        "tolerant_parser": base.groupby("causal_level")["correct_parsed"].mean(),
        "n": base.groupby("causal_level").size(),
    }
)
baseline.loc["overall"] = [base["correct_exact"].mean(), base["correct_parsed"].mean(), len(base)]
baseline

In [ ]:
parsed = base[base["status"] == "parsed"]
display(rate_table(parsed, ["causal_level"], flag="correct_parsed"))
display(rate_table(parsed, ["story_kind"], flag="correct_parsed"))
print(f"Yes rate among parsed answers: {parsed['is_yes'].mean():.3f} (answers are 50/50)")

## B · Commonsense alignment

In [ ]:
result_paths = [p for p in sorted(INTERIM.glob("logprobs-*.jsonl")) if not p.name.endswith("-smoke.jsonl")]
print("\n".join(p.name for p in result_paths))
df = load_scored(INTERIM / "variants-sample.jsonl", result_paths)
df["is_yes"] = df["pred"] == "yes"
df.groupby("model").agg(
    items=("item_id", "size"),
    median_answer_mass=("answer_mass", "median"),
    yes_rate=("is_yes", "mean"),
)

### Accuracy by variant

In [ ]:
acc = rate_table(df, ["model", "variant"])
acc.pivot(index="model", columns="variant", values="rate")

### H1 · Anticommonsense gap on matched pairs, and nonsense gap

In [ ]:
h1_paired = paired_accuracy_gap(df)
h1_paired

In [ ]:
h1_nonsense = nonsense_gap(df)
h1_nonsense

### H1b · P(Yes) shift on matched relational items (commonsense minus anticommonsense)

In [ ]:
h1b = p_yes_shift(df)
h1b

### H2 · Accuracy by rung

In [ ]:
h2 = rate_table(df, ["model", "rung"])
h2.pivot(index="model", columns="rung", values="rate")

### Answer bias and topology

In [ ]:
yes_rates = rate_table(df, ["model", "variant"], flag="is_yes")
display(yes_rates.pivot(index="model", columns="variant", values="rate"))
rate_table(df, ["model", "graph_id"]).pivot(index="graph_id", columns="model", values="rate")

## C · Figures and table for the paper

In [ ]:
order = ["commonsense", "noncommonsense", "anticommonsense"]
models = sorted(df["model"].unique())
fig, ax = plt.subplots(figsize=(6.5, 3))
width = 0.8 / len(order)
for i, variant in enumerate(order):
    rows = acc[acc["variant"] == variant].set_index("model").reindex(models)
    x = np.arange(len(models)) + (i - 1) * width
    ax.bar(x, rows["rate"], width, label=variant,
           yerr=[rows["rate"] - rows["lo"], rows["hi"] - rows["rate"]], capsize=2)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_xticks(np.arange(len(models)), [short(m) for m in models], rotation=15, fontsize=7)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "accuracy_by_variant.pdf")

In [ ]:
for model in models:
    nodes, edges = failure_graph(df[df["model"] == model])
    fig, ax = plt.subplots(figsize=(7, 4))
    draw_failure_graph(nodes, edges, ax=ax, title=short(model))
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"failure_graph_{short(model)}.pdf")
    display(edges.sort_values("lift", ascending=False).head(8))

In [ ]:
def cell(row):
    return f"{row.rate:.2f} [{row.lo:.2f}, {row.hi:.2f}]"


shift_yes = h1b[h1b["answer"] == "yes"].set_index("model")
lines = []
for model in models:
    cells = {r.variant: cell(r) for r in acc[acc["model"] == model].itertuples()}
    p_yes = (
        f"{shift_yes.loc[model, 'p_yes_common']:.2f} / {shift_yes.loc[model, 'p_yes_anti']:.2f}"
        if model in shift_yes.index
        else "--"
    )
    name = short(model).replace("_", r"\_")
    lines.append(
        f"{name} & {cells.get('commonsense', '--')} & {cells.get('noncommonsense', '--')} & "
        f"{cells.get('anticommonsense', '--')} & {p_yes} \\\\"
    )
(TAB_DIR / "variants_rows.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))